# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [1]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint16, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

Vocabulary length: 194263
Tokens length: 100000404


## Split dataset into training and test

In [2]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [3]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

## Traing loop

In [4]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 128
vocabulary_size = len(stoi)
block_size = 64
batch_size = 16
block_layers = 4
gradient = Adam(lr=3e-6)

model = MiniGPT(vocabulary_size, d_model, block_size, block_layers, gradient)
# model = MiniGPT.__new__(MiniGPT)
# model = model.load("saved_model")
xb, yb = get_batch("train", block_size, batch_size)

for step in range(4880):
    xb, yb = get_batch("train", block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)
    
    model.backward()

    if step % 10 == 0:
        print(f"saved model on step {step}")
        model.save("saved_model")

    if step % 50 == 0:
        print("step:", step, "loss:", loss)

saved model on step 0
step: 0 loss: 12.045675
saved model on step 10
saved model on step 20
saved model on step 30
saved model on step 40
saved model on step 50
step: 50 loss: 7.540022
saved model on step 60
saved model on step 70
saved model on step 80
saved model on step 90
saved model on step 100
step: 100 loss: 4.394108
saved model on step 110
saved model on step 120
saved model on step 130
saved model on step 140
saved model on step 150
step: 150 loss: 2.6490216
saved model on step 160
saved model on step 170
saved model on step 180
saved model on step 190
saved model on step 200
step: 200 loss: 2.478572
saved model on step 210
saved model on step 220
saved model on step 230
saved model on step 240
saved model on step 250
step: 250 loss: 2.295598
saved model on step 260
saved model on step 270
saved model on step 280
saved model on step 290
saved model on step 300
step: 300 loss: 2.3586962
saved model on step 310
saved model on step 320
saved model on step 330
saved model on step 

In [4]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        
        logits, _ = model.forward(idx_cond, np.array([1]))
        
        logits = logits[:, -1, :]
        
        max_logits = np.max(logits, axis=-1, keepdims=True)
        exp_logits = np.exp(logits - max_logits)
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
        
        next_tokens = []
        for b in range(probs.shape[0]):
            next_token = np.random.choice(len(stoi), p=probs[b])
            next_tokens.append(next_token)
        
        next_token = np.array(next_tokens).reshape(-1, 1)
        
        idx = np.concatenate([idx, next_token], axis=1)
    
    return idx

In [ ]:
from NoTorchAI.LLM.MiniGPT import MiniGPT


# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")

itos = {i: ch for ch, i in stoi.items()}

prompt = "What is ur purpose"
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.int64)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 100)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

What is ur purpose     must     turbines   and    Simon   1905  tightlySenjō   to   worked  presented   to   on   aftermath   or   disadvantagedSenjō  into   writing   nearly   given   the   earliest   miles   (   Qingtongqi   in
